# EEG Feature Extraction - Hierarchical Strategies

This notebook extracts EEG features from preprocessed EEG data using a **principled granularity hierarchy**.

**Parameterized**: Set `STRATEGY` to choose feature extraction level.

**Run this notebook ONCE per strategy after EEG preprocessing completes.**

All other model notebooks will load the saved features instead of re-extracting.

---

## Feature Extraction Hierarchy

| Level | Strategy | Features | Description | Builds On |
|-------|----------|----------|-------------|----------|
| 0 | `channels_raw` | 20 | Total power per electrode | Base |
| 1 | `regional_raw` | 4 | Regional averages of total power | channels_raw |
| 2 | `channels_bands` | 80 | 20 channels × 4 frequency bands | channels_raw |
| 3 | `regional_bands` | 16 | 4 regions × 4 frequency bands | channels_bands |
| 4 | `extended` | 160 | channels_bands + lateralization + temporal | channels_bands |

Each level builds on the previous, allowing systematic ablation studies:
- Does band decomposition help? Compare Level 0 vs Level 2
- Does regional aggregation help? Compare Level 0 vs Level 1, or Level 2 vs Level 3
- Do derived features help? Compare Level 2 vs Level 4

In [37]:
# ============================================================================
# CONFIGURATION: Set extraction strategy
# ============================================================================
STRATEGY = 'extended'  # Options: 'channels_raw', 'regional_raw', 'channels_bands', 'regional_bands', 'extended'
# ============================================================================

import sys
sys.path.append('../..')

import pickle
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from src.features.eeg_features import (
    extract_eeg_features, 
    get_feature_metadata,
    get_strategy_hierarchy,
    CHANNEL_NAMES,
    CHANNEL_REGIONS,
    FREQ_BANDS
)

# Display hierarchy
hierarchy = get_strategy_hierarchy()
print(f"\n{'='*70}")
print(f"EEG FEATURE EXTRACTION: {STRATEGY.upper()} (Level {hierarchy[STRATEGY]['level']})")
print(f"{'='*70}\n")
print(f"Feature extraction started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nStrategy Hierarchy:")
for strat, info in hierarchy.items():
    marker = "-->" if strat == STRATEGY else "   "
    print(f"  {marker} Level {info['level']}: {strat:20s} ({info['n_features']:3d} features)")


EEG FEATURE EXTRACTION: EXTENDED (Level 4)

Feature extraction started: 2026-03-30 09:48:26

Strategy Hierarchy:
      Level 0: channels_raw         ( 20 features)
      Level 1: regional_raw         (  4 features)
      Level 2: channels_bands       ( 80 features)
      Level 3: regional_bands       ( 16 features)
  --> Level 4: extended             (112 features)


## 1. Load Preprocessed EEG Data

Load the preprocessed EEG pickle file containing display_eeg data.

In [38]:
eeg_data_path = '../../data/eeg/Copy of preprocessed_eeg.pkl'

print(f"Loading EEG data from: {eeg_data_path}")
with open(eeg_data_path, 'rb') as f:
    eeg_df = pickle.load(f)

print(f"\n✓ Loaded {len(eeg_df)} trials")
print(f"  Unique subjects: {eeg_df['subject_date_id'].nunique()}")

# Check EEG data structure
sample_eeg = eeg_df['display_eeg'].iloc[0]
print(f"  EEG array shape: {sample_eeg.shape} (time × channels)")
print(f"\nColumns: {eeg_df.columns.tolist()}")

Loading EEG data from: ../../data/eeg/Copy of preprocessed_eeg.pkl

✓ Loaded 10855 trials
  Unique subjects: 85
  EEG array shape: (320, 20) (time × channels)

Columns: ['subject_date_id', 'trial_id', 'display_eeg']


## 2. Channel and Region Configuration

Display the channel configuration used for feature extraction.

In [39]:
print("\n" + "="*70)
print("CHANNEL CONFIGURATION (from chan_locs.sfp)")
print("="*70)

print(f"\n20 EEG Channels (10-20 system):")
print(f"  {', '.join(CHANNEL_NAMES)}")

print(f"\n4 Brain Regions:")
for region, channels in CHANNEL_REGIONS.items():
    print(f"  {region:10s}: {', '.join(channels)}")

print(f"\n4 Frequency Bands:")
for band, (f_low, f_high) in FREQ_BANDS.items():
    print(f"  {band:6s}: {f_low:4.1f} - {f_high:4.1f} Hz")


CHANNEL CONFIGURATION (from chan_locs.sfp)

20 EEG Channels (10-20 system):
  Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2

4 Brain Regions:
  Frontal   : Fp1, Fp2, F7, F3, Fz, F4, F8
  Central   : T3, C3, Cz, C4, T4
  Parietal  : T5, P3, Pz, P4, T6
  Occipital : O1, POz, O2

4 Frequency Bands:
  Delta :  0.5 -  4.0 Hz
  Theta :  4.0 -  8.0 Hz
  Alpha :  8.0 - 13.0 Hz
  Beta  : 13.0 - 30.0 Hz


## 3. Extract EEG Features

Extract features based on selected strategy:

### Level 0: channels_raw (20 features)
- Total (broadband) power per electrode
- **Features:** `eeg_{channel}` for each of 20 channels
- **Use case:** Simplest baseline, tests if spatial pattern alone carries signal

### Level 1: regional_raw (4 features)
- Regional averages of total power
- **Features:** `eeg_{Frontal,Central,Parietal,Occipital}`
- **Use case:** Most compact representation, tests regional differences

### Level 2: channels_bands (80 features)
- Band power per channel (20 channels × 4 bands)
- **Features:** `eeg_{band}_{channel}`
- **Use case:** Full spatial + spectral resolution

### Level 3: regional_bands (16 features)
- Regional average band power (4 regions × 4 bands)
- **Features:** `eeg_{band}_{region}`
- **Use case:** Standard approach, balances detail with interpretability

### Level 4: extended (160 features)
- channels_bands + temporal dynamics + lateralization
- **Channel-band power:** 80 features
- **Temporal dynamics:** `eeg_{band}_{region}_{mean,std,slope}` (48 features)
- **Lateralization:** `eeg_{band}_{pair}_lateralization` (32 features)
- **Use case:** Maximum feature richness for best performance

In [40]:
print(f"\nExtracting EEG features using '{STRATEGY}' strategy...\n")

eeg_features_df = extract_eeg_features(
    eeg_df=eeg_df,
    strategy=STRATEGY,
    fs=256,
    verbose=True
)

# Get feature columns
eeg_cols = [c for c in eeg_features_df.columns if c.startswith('eeg_')]

print(f"\n{'='*70}")
print(f"✓ Extracted {len(eeg_cols)} EEG features")
print(f"  Example features: {eeg_cols[:5]}")
if len(eeg_cols) > 5:
    print(f"  ... and {len(eeg_cols) - 5} more")
print(f"{'='*70}")


Extracting EEG features using 'extended' strategy...

Extracting EEG features using 'extended' strategy (Level 4)...
  Sampling rate: 256 Hz
  Trials: 10855
✓ Extracted 160 EEG features
  Channel-band power: 80 features
  Temporal dynamics: 48 features
  Lateralization: 32 features

✓ Extracted 160 EEG features
  Example features: ['eeg_Delta_Fp1', 'eeg_Delta_F7', 'eeg_Delta_F8', 'eeg_Delta_T4', 'eeg_Delta_T6']
  ... and 155 more


## 4. Inspect Features

In [41]:
# Display sample data
print("\nSample EEG features:")
display(eeg_features_df.head(3))

# Feature statistics
print("\nFeature statistics:")
display(eeg_features_df[eeg_cols].describe())


Sample EEG features:


,subject_id,trial_id,eeg_Delta_Fp1,eeg_Delta_F7,eeg_Delta_F8,eeg_Delta_T4,eeg_Delta_T6,eeg_Delta_T5,eeg_Delta_T3,eeg_Delta_Fp2,...,eeg_Beta_Frontal_slope,eeg_Beta_Central_mean,eeg_Beta_Central_std,eeg_Beta_Central_slope,eeg_Beta_Parietal_mean,eeg_Beta_Parietal_std,eeg_Beta_Parietal_slope,eeg_Beta_Occipital_mean,eeg_Beta_Occipital_std,eeg_Beta_Occipital_slope
0,0831_1300_9M4VCHG,0_0831_1300_9M4VCHG,0.000605,0.001861,0.006466,0.003068,0.005012,0.000499,0.002324,0.001070,...,0.000001,0.000447,0.000143,0.000169,0.000230,0.000057,0.000068,0.000118,0.000035,0.000037
1,0831_1300_9M4VCHG,1_0831_1300_9M4VCHG,0.000483,0.001588,0.005634,0.000917,0.004017,0.000299,0.002791,0.004292,...,0.000136,0.001631,0.000560,-0.000215,0.000543,0.000110,0.000119,0.000499,0.000133,-0.000054
2,0831_1300_9M4VCHG,2_0831_1300_9M4VCHG,0.003610,0.002319,0.001476,0.003023,0.002238,0.001336,0.007630,0.009303,...,-0.000116,0.000505,0.000120,-0.000143,0.000239,0.000062,-0.000074,0.000126,0.000051,-0.000062



Feature statistics:


,eeg_Delta_Fp1,eeg_Delta_F7,eeg_Delta_F8,eeg_Delta_T4,eeg_Delta_T6,eeg_Delta_T5,eeg_Delta_T3,eeg_Delta_Fp2,eeg_Delta_O1,eeg_Delta_P3,...,eeg_Beta_Frontal_slope,eeg_Beta_Central_mean,eeg_Beta_Central_std,eeg_Beta_Central_slope,eeg_Beta_Parietal_mean,eeg_Beta_Parietal_std,eeg_Beta_Parietal_slope,eeg_Beta_Occipital_mean,eeg_Beta_Occipital_std,eeg_Beta_Occipital_slope
count,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,...,10855.000000,10855.000000,1.085500e+04,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,1.085500e+04,10855.000000
mean,0.111851,0.112327,0.123142,0.100004,0.108478,0.115377,0.108827,0.117383,0.119894,0.105600,...,-0.004570,0.054306,2.351718e-02,-0.006155,0.048451,0.019375,-0.006424,0.043950,1.901458e-02,-0.006352
std,0.509233,0.524628,0.565807,0.498324,0.577381,0.480556,0.520107,0.530036,0.454151,0.597114,...,0.096968,0.254938,1.656723e-01,0.182619,0.110970,0.069081,0.065659,0.089874,5.558720e-02,0.053147
min,0.000012,0.000017,0.000023,0.000015,0.000008,0.000003,0.000007,0.000015,0.000004,0.000005,...,-7.242276,0.000027,1.561821e-07,-12.581209,0.000020,0.000001,-2.598357,0.000021,6.411233e-07,-3.312181
25%,0.001941,0.002232,0.002501,0.001997,0.002137,0.001592,0.001750,0.002639,0.001564,0.000884,...,-0.003745,0.000617,1.407351e-04,-0.004132,0.000520,0.000121,-0.005292,0.000523,1.357621e-04,-0.004665
50%,0.015214,0.016040,0.015197,0.012702,0.012873,0.013241,0.013481,0.016899,0.011460,0.008682,...,-0.000065,0.005124,1.532646e-03,-0.000061,0.005837,0.001591,-0.000065,0.004861,1.553754e-03,-0.000075
75%,0.119755,0.107534,0.122054,0.094512,0.115013,0.137057,0.113297,0.125172,0.131039,0.099818,...,0.000259,0.063173,1.615622e-02,0.000160,0.069174,0.018331,0.000110,0.067409,1.968850e-02,0.000115
max,36.801483,36.801483,36.801483,36.801483,36.801483,36.801483,36.801483,36.801483,36.801483,36.801483,...,2.400682,17.788908,1.079147e+01,8.027274,4.492504,2.696434,1.507368,4.651772,2.836602e+00,0.879458


## 5. Categorize Features by Type

Group features by their type based on the extraction strategy.

In [42]:
feature_categories = {}

if STRATEGY == 'channels_raw':
    feature_categories['channel_power'] = eeg_cols
    print(f"\nChannel power features: {len(feature_categories['channel_power'])}")
    print(f"  Channels: {[c.replace('eeg_', '') for c in eeg_cols]}")

elif STRATEGY == 'regional_raw':
    feature_categories['regional_power'] = eeg_cols
    print(f"\nRegional power features: {len(feature_categories['regional_power'])}")
    print(f"  Regions: {[c.replace('eeg_', '') for c in eeg_cols]}")

elif STRATEGY == 'channels_bands':
    for band in FREQ_BANDS.keys():
        band_cols = [c for c in eeg_cols if f'eeg_{band}_' in c]
        feature_categories[f'{band}_channels'] = band_cols
    print(f"\nChannel-band power features by band:")
    for band, cols in feature_categories.items():
        print(f"  {band}: {len(cols)} features")

elif STRATEGY == 'regional_bands':
    for band in FREQ_BANDS.keys():
        band_cols = [c for c in eeg_cols if f'eeg_{band}_' in c]
        feature_categories[f'{band}_regional'] = band_cols
    print(f"\nRegional-band power features by band:")
    for band, cols in feature_categories.items():
        print(f"  {band}: {len(cols)} features")

elif STRATEGY == 'extended':
    # Separate by feature type
    channel_band_cols = [c for c in eeg_cols if not any(x in c for x in 
                        ['_mean', '_std', '_slope', '_lateralization'])]
    temporal_cols = [c for c in eeg_cols if any(x in c for x in 
                    ['_mean', '_std', '_slope'])]
    lat_cols = [c for c in eeg_cols if '_lateralization' in c]
    
    feature_categories['channel_band_power'] = channel_band_cols
    feature_categories['temporal_dynamics'] = temporal_cols
    feature_categories['lateralization'] = lat_cols
    
    print(f"\nExtended features by type:")
    print(f"  Channel-band power: {len(channel_band_cols)} features")
    print(f"  Temporal dynamics: {len(temporal_cols)} features")
    print(f"  Lateralization: {len(lat_cols)} features")

print(f"\nTotal features: {len(eeg_cols)}")


Extended features by type:
  Channel-band power: 80 features
  Temporal dynamics: 48 features
  Lateralization: 32 features

Total features: 160


## 6. Prepare Metadata

Create comprehensive metadata for reproducibility.

In [43]:
metadata = get_feature_metadata(STRATEGY)
metadata.update({
    'extraction_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_trials': len(eeg_features_df),
    'n_subjects': eeg_features_df['subject_id'].nunique(),
    'input_file': eeg_data_path,
    'description': f'EEG features extracted using {STRATEGY} strategy (Level {metadata["level"]})'
})

print("\nMetadata:")
for key, value in metadata.items():
    if not isinstance(value, (dict, list)):
        print(f"  {key}: {value}")


Metadata:
  strategy: extended
  level: 4
  builds_on: channels_bands
  sampling_rate: 256
  n_channels: 20
  n_regions: 4
  n_features: 160
  extraction_date: 2026-03-30 09:51:14
  n_trials: 10855
  n_subjects: 85
  input_file: ../../data/eeg/Copy of preprocessed_eeg.pkl
  description: EEG features extracted using extended strategy (Level 4)


## 7. Save to Pickle File

Save features with metadata for use in other notebooks.

In [44]:
# Output path based on strategy
output_dir = Path('../../data/features')
output_dir.mkdir(parents=True, exist_ok=True)

# Strategy-to-filename mapping
strategy_filenames = {
    'channels_raw': 'eeg_features_channels_raw.pkl',
    'regional_raw': 'eeg_features_regional_raw.pkl',
    'channels_bands': 'eeg_features_channels_bands.pkl',
    'regional_bands': 'eeg_features_regional_bands.pkl',
    'extended': 'eeg_features_extended.pkl'
}

output_filename = strategy_filenames[STRATEGY]
output_path = output_dir / output_filename

# Prepare output data (matching main feature extraction format)
output_data = {
    'eeg_features_df': eeg_features_df,
    'feature_columns': eeg_cols,
    'feature_categories': feature_categories,
    'metadata': metadata
}

# Save
with open(output_path, 'wb') as f:
    pickle.dump(output_data, f)

file_size_mb = output_path.stat().st_size / 1024 / 1024

print(f"\n{'='*70}")
print(f"✓ EEG features saved to: {output_path}")
print(f"  File size: {file_size_mb:.2f} MB")
print(f"  Trials: {len(eeg_features_df)}")
print(f"  Subjects: {eeg_features_df['subject_id'].nunique()}")
print(f"  Features: {len(eeg_cols)}")
print(f"\nCompleted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")


✓ EEG features saved to: ../../data/features/eeg_features_extended.pkl
  File size: 13.52 MB
  Trials: 10855
  Subjects: 85
  Features: 160

Completed: 2026-03-30 09:51:14


## 8. Verification

Verify the saved file can be loaded correctly.

In [45]:
# Test loading
print("\nVerifying saved file...")
with open(output_path, 'rb') as f:
    test_data = pickle.load(f)

print(f"✓ File loads successfully")
print(f"  Keys: {list(test_data.keys())}")
print(f"  Features: {len(test_data['feature_columns'])}")
print(f"  Trials: {len(test_data['eeg_features_df'])}")
print(f"  Strategy: {test_data['metadata']['strategy']}")
print(f"  Level: {test_data['metadata']['level']}")
print("\n✓ Verification complete!")


Verifying saved file...
✓ File loads successfully
  Keys: ['eeg_features_df', 'feature_columns', 'feature_categories', 'metadata']
  Features: 160
  Trials: 10855
  Strategy: extended
  Level: 4

✓ Verification complete!


---

## Usage in Other Notebooks

To use these EEG features in fusion models or other analyses:

```python
import pickle

# Load EEG features (choose the strategy you want)
strategy = 'channels_raw'  # or 'regional_raw', 'channels_bands', 'regional_bands', 'extended'
with open(f'../../data/features/eeg_features_{strategy}.pkl', 'rb') as f:
    eeg_data = pickle.load(f)

eeg_features_df = eeg_data['eeg_features_df']
eeg_cols = eeg_data['feature_columns']
metadata = eeg_data['metadata']

print(f"Loaded {metadata['strategy']} (Level {metadata['level']}): {len(eeg_cols)} features")

# Merge with other modalities
merged_df = merged_df.merge(
    eeg_features_df,
    on=['subject_id', 'trial_id'],
    how='inner'
)
```

---

## Strategy Hierarchy Reference

| Level | Strategy | Features | Builds On | Output File |
|-------|----------|----------|-----------|-------------|
| 0 | `channels_raw` | 20 | Base | `eeg_features_channels_raw.pkl` |
| 1 | `regional_raw` | 4 | channels_raw | `eeg_features_regional_raw.pkl` |
| 2 | `channels_bands` | 80 | channels_raw | `eeg_features_channels_bands.pkl` |
| 3 | `regional_bands` | 16 | channels_bands | `eeg_features_regional_bands.pkl` |
| 4 | `extended` | 160 | channels_bands | `eeg_features_extended.pkl` |

---

## Channel Configuration

All features use standardized channel configuration from `data/eeg/chan_locs.sfp`:

**20 EEG Channels (10-20 system):**
```
Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2
```

**4 Brain Regions:**
- **Frontal:** Fp1, Fp2, F7, F3, Fz, F4, F8
- **Central:** T3, C3, Cz, C4, T4
- **Parietal:** T5, P3, Pz, P4, T6
- **Occipital:** O1, POz, O2

**4 Frequency Bands:**
- Delta: 0.5-4 Hz
- Theta: 4-8 Hz
- Alpha: 8-13 Hz
- Beta: 13-30 Hz